# Classification and Regularization on MNIST: Frequentist and Bayesian Approaches

**STAT646 Midterm Project**

**Author:** [Your Name]

**Date:** April 2026

---

This project explores classification and regularization from both frequentist and Bayesian viewpoints using MNIST handwritten digit data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    log_loss, precision_recall_fscore_support
)
from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA

import pymc as pm
import arviz as az

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

print("Setup complete.")

---
## Part A: Data Description and Setup

We use a subset of MNIST (5000 samples, all 10 classes) to keep computation manageable while retaining multiclass complexity.

In [ ]:
# Load MNIST
print("Loading MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_full, y_full = mnist.data, mnist.target.astype(int)

# Subsample
N_SAMPLES = 5000
indices = np.random.choice(len(X_full), size=N_SAMPLES, replace=False)
X, y = X_full[indices], y_full[indices]

# Train/Test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Train/Val split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, stratify=y_train_val, random_state=RANDOM_STATE
)

print(f"\nDataset Summary:")
print(f"  Total samples: {N_SAMPLES}")
print(f"  Classes: {len(np.unique(y))} (digits 0-9)")
print(f"  Features: {X.shape[1]} (28x28 pixels)")
print(f"  Pixel range: [{X.min()}, {X.max()}]")
print(f"\nSplits:")
print(f"  Train: {len(X_train)} ({len(X_train)/N_SAMPLES*100:.1f}%)")
print(f"  Val:   {len(X_val)} ({len(X_val)/N_SAMPLES*100:.1f}%)")
print(f"  Test:  {len(X_test)} ({len(X_test)/N_SAMPLES*100:.1f}%)")

In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    ax = axes[digit // 5, digit % 5]
    ax.imshow(X_train[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'Digit: {digit}')
    ax.axis('off')
plt.suptitle('Sample Digits from MNIST', fontsize=14)
plt.tight_layout()
plt.show()

### Preprocessing

1. **StandardScaler**: Standardize pixels to mean=0, std=1
2. **Pipeline**: Ensures preprocessing is fit only on training data (no leakage)
3. **Cross-validation**: Hyperparameter tuning on training set only; test set held out

### Multinomial Logistic Regression Model

The probability of class $k$ given features $\mathbf{x}$ is:

$$P(Y = k | \mathbf{X} = \mathbf{x}) = \frac{\exp(\alpha_k + \mathbf{x}^\top \boldsymbol{\beta}_k)}{\sum_{j=1}^{K} \exp(\alpha_j + \mathbf{x}^\top \boldsymbol{\beta}_j)}$$

This is a **probabilistic classifier** because:
- Outputs valid probabilities summing to 1
- Log-odds are linear in features
- Quantifies prediction uncertainty

**Prediction**: $\hat{y} = \arg\max_k P(Y = k | \mathbf{x})$

---
## Part B: Frequentist Classification and Regularization

We fit and compare:
1. Multinomial logistic regression (no penalty)
2. Ridge-penalized ($\lambda \|\boldsymbol{\beta}\|_2^2$)
3. Lasso-penalized ($\lambda \|\boldsymbol{\beta}\|_1$)
4. Elastic Net (bonus: combines L1 + L2)
5. Linear SVM
6. RBF SVM (optional extension)

In [ ]:
# X_train_val is already assembled in Part A (train+val).
# We reference it directly here — no need to reassemble.
results = {}
models = {}

In [ ]:
# (i) Unpenalized Logistic Regression
# C=np.inf is equivalent to no regularization (replaces deprecated penalty=None)
print("Fitting: Multinomial Logistic Regression (No Penalty)")
t0 = time()
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(C=np.inf, solver='lbfgs', max_iter=1000,
                                random_state=RANDOM_STATE))
])
pipe_lr.fit(X_train_val, y_train_val)
train_time = time() - t0

y_pred = pipe_lr.predict(X_test)
y_prob = pipe_lr.predict_proba(X_test)

results['Logistic (No Penalty)'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'log_loss': log_loss(y_test, y_prob),
    'train_time': train_time,
    'best_params': None
}
models['Logistic (No Penalty)'] = pipe_lr

print(f"  Accuracy: {results['Logistic (No Penalty)']['accuracy']:.4f}")
print(f"  Log-loss: {results['Logistic (No Penalty)']['log_loss']:.4f}")
print(f"  Time: {train_time:.2f}s")

In [ ]:
# (ii) Ridge-Penalized Logistic Regression
print("Fitting: Ridge Logistic Regression")
print("  Penalty: lambda * ||beta||_2^2")

t0 = time()
pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE))
])

grid_ridge = GridSearchCV(
    pipe_ridge, 
    {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_ridge.fit(X_train_val, y_train_val)
train_time = time() - t0

y_pred = grid_ridge.predict(X_test)
y_prob = grid_ridge.predict_proba(X_test)

results['Ridge'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'log_loss': log_loss(y_test, y_prob),
    'train_time': train_time,
    'best_params': grid_ridge.best_params_,
    'cv_score': grid_ridge.best_score_
}
models['Ridge'] = grid_ridge

print(f"  Best C (1/lambda): {grid_ridge.best_params_['clf__C']}")
print(f"  CV Accuracy: {grid_ridge.best_score_:.4f}")
print(f"  Test Accuracy: {results['Ridge']['accuracy']:.4f}")
print(f"  Log-loss: {results['Ridge']['log_loss']:.4f}")

In [ ]:
# (iii) Lasso-Penalized Logistic Regression
print("Fitting: Lasso Logistic Regression")
print("  Penalty: lambda * ||beta||_1")

t0 = time()
pipe_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=RANDOM_STATE))
])

grid_lasso = GridSearchCV(
    pipe_lasso,
    {'clf__C': [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_lasso.fit(X_train_val, y_train_val)
train_time = time() - t0

y_pred = grid_lasso.predict(X_test)
y_prob = grid_lasso.predict_proba(X_test)

results['Lasso'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'log_loss': log_loss(y_test, y_prob),
    'train_time': train_time,
    'best_params': grid_lasso.best_params_,
    'cv_score': grid_lasso.best_score_
}
models['Lasso'] = grid_lasso

print(f"  Best C: {grid_lasso.best_params_['clf__C']}")
print(f"  Test Accuracy: {results['Lasso']['accuracy']:.4f}")
print(f"  Log-loss: {results['Lasso']['log_loss']:.4f}")

In [ ]:
# (Bonus) Elastic Net
print("Fitting: Elastic Net Logistic Regression")
print("  Penalty: lambda1 * ||beta||_1 + lambda2 * ||beta||_2^2")

t0 = time()
pipe_enet = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=1000, random_state=RANDOM_STATE))
])

grid_enet = GridSearchCV(
    pipe_enet,
    {'clf__C': [0.01, 0.1, 1, 10], 'clf__l1_ratio': [0.25, 0.5, 0.75]},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_enet.fit(X_train_val, y_train_val)
train_time = time() - t0

y_pred = grid_enet.predict(X_test)
y_prob = grid_enet.predict_proba(X_test)

results['Elastic Net'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'log_loss': log_loss(y_test, y_prob),
    'train_time': train_time,
    'best_params': grid_enet.best_params_,
    'cv_score': grid_enet.best_score_
}
models['Elastic Net'] = grid_enet

print(f"  Best params: {grid_enet.best_params_}")
print(f"  Test Accuracy: {results['Elastic Net']['accuracy']:.4f}")

In [ ]:
# (iv) Linear SVM
print("Fitting: Linear SVM")

t0 = time()
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(max_iter=2000, dual='auto', random_state=RANDOM_STATE))
])

grid_svm = GridSearchCV(
    pipe_svm,
    {'clf__C': [0.001, 0.01, 0.1, 1, 10]},
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_svm.fit(X_train_val, y_train_val)
train_time = time() - t0

y_pred = grid_svm.predict(X_test)

results['Linear SVM'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'log_loss': None,
    'train_time': train_time,
    'best_params': grid_svm.best_params_,
    'cv_score': grid_svm.best_score_
}
models['Linear SVM'] = grid_svm

print(f"  Best C: {grid_svm.best_params_['clf__C']}")
print(f"  Test Accuracy: {results['Linear SVM']['accuracy']:.4f}")

In [ ]:
# (Optional) RBF SVM
print("Fitting: RBF SVM")

t0 = time()
pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', random_state=RANDOM_STATE))
])

grid_rbf = GridSearchCV(
    pipe_rbf,
    {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.01, 0.001]},
    cv=3, scoring='accuracy', n_jobs=-1
)
grid_rbf.fit(X_train_val, y_train_val)
train_time = time() - t0

y_pred = grid_rbf.predict(X_test)

results['RBF SVM'] = {
    'accuracy': accuracy_score(y_test, y_pred),
    'log_loss': None,
    'train_time': train_time,
    'best_params': grid_rbf.best_params_,
    'cv_score': grid_rbf.best_score_
}
models['RBF SVM'] = grid_rbf

print(f"  Best params: {grid_rbf.best_params_}")
print(f"  Test Accuracy: {results['RBF SVM']['accuracy']:.4f}")

### Results Summary Table

In [ ]:
df_results = pd.DataFrame(results).T
df_results = df_results[['accuracy', 'log_loss', 'train_time', 'cv_score', 'best_params']]
df_results.columns = ['Test Accuracy', 'Log-Loss', 'Train Time (s)', 'CV Score', 'Best Params']
df_results

### Coefficient Analysis

In [ ]:
# Analyze sparsity and shrinkage
coef_analysis = {}

for name in ['Ridge', 'Lasso', 'Elastic Net']:
    clf = models[name].best_estimator_.named_steps['clf']
    coefs = clf.coef_
    coef_analysis[name] = {
        'mean_abs': np.mean(np.abs(coefs)),
        'max_abs': np.max(np.abs(coefs)),
        'n_nonzero': np.sum(np.abs(coefs) > 1e-10),
        'total': coefs.size,
        'sparsity': 1 - np.sum(np.abs(coefs) > 1e-10) / coefs.size,
        'coefs': coefs
    }

print("Coefficient Statistics:")
print("-" * 65)
print(f"{'Model':<15} {'Mean |coef|':<12} {'Max |coef|':<12} {'Non-zero':<10} {'Sparsity %':<10}")
print("-" * 65)
for name, stats in coef_analysis.items():
    print(f"{name:<15} {stats['mean_abs']:<12.4f} {stats['max_abs']:<12.4f} "
          f"{stats['n_nonzero']:<10} {stats['sparsity']*100:<10.2f}")

In [ ]:
# Visualize coefficients as images
# Note: pixels were StandardScaler-transformed, so low-variance background pixels
# can dominate after scaling. Spatial patterns are qualitatively valid.
fig, axes = plt.subplots(3, 10, figsize=(15, 5))

for row, (name, stats) in enumerate(coef_analysis.items()):
    coefs = stats['coefs']
    for digit in range(10):
        ax = axes[row, digit]
        coef_img = coefs[digit].reshape(28, 28)
        vmax = np.max(np.abs(coef_img))
        ax.imshow(coef_img, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        # Hide ticks/spines without axis('off'), which also removes set_ylabel
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if digit == 0:
            # ax.text survives after we suppress ticks
            ax.text(-0.15, 0.5, name, fontsize=10, rotation=0,
                    ha='right', va='center', transform=ax.transAxes)
        if row == 0:
            ax.set_title(f'{digit}', fontsize=10)

plt.suptitle('Coefficient Maps by Digit (Red=Positive, Blue=Negative)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for idx, (name, model) in enumerate(list(models.items())[:6]):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=range(10), yticklabels=range(10))
    axes[idx].set_title(f'{name}\nAcc: {accuracy_score(y_test, y_pred):.3f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class performance
y_pred = models['Ridge'].predict(X_test)
print("Classification Report (Ridge):")
print(classification_report(y_test, y_pred, digits=4))

---
## Part C: Bayesian Classification and Shrinkage

We use a binary subset (digits 3 vs 8) and fit Bayesian logistic regression with:
1. Gaussian prior (Ridge analogue)
2. Laplace prior (Lasso analogue)
3. Horseshoe prior (optional extension)

### Connection: Frequentist Regularization <-> Bayesian Priors

| Frequentist | Bayesian | Interpretation |
|-------------|----------|----------------|
| Ridge ($\lambda\|\beta\|_2^2$) | Gaussian prior: $\beta \sim N(0, \sigma^2)$ | MAP = Ridge solution |
| Lasso ($\lambda\|\beta\|_1$) | Laplace prior: $\beta \sim \text{Laplace}(0, b)$ | MAP = Lasso solution |

In [ ]:
# Prepare binary subset
# Use X_train_val (full train set) so Bayesian models see as much data
# as the frequentist models in Part B — consistent comparison
DIGIT1, DIGIT2 = 3, 8

train_mask = (y_train_val == DIGIT1) | (y_train_val == DIGIT2)
test_mask  = (y_test == DIGIT1)      | (y_test == DIGIT2)

X_train_bin = X_train_val[train_mask]
y_train_bin = (y_train_val[train_mask] == DIGIT2).astype(int)

X_test_bin = X_test[test_mask]
y_test_bin = (y_test[test_mask] == DIGIT2).astype(int)

print(f"Binary classification: {DIGIT1} vs {DIGIT2}")
print(f"Train: {len(y_train_bin)}, Test: {len(y_test_bin)}")

In [ ]:
# PCA for computational tractability
N_COMPONENTS = 30

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bin)
X_test_scaled = scaler.transform(X_test_bin)

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.3f}")

In [ ]:
bayesian_results = {}

In [ ]:
# (i) Gaussian Prior
print("Fitting: Bayesian Logistic Regression with Gaussian Prior")

with pm.Model() as model_gaussian:
    intercept = pm.Normal('intercept', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=1, shape=N_COMPONENTS)
    eta = intercept + pm.math.dot(X_train_pca, beta)
    p = pm.math.sigmoid(eta)
    y_obs = pm.Bernoulli('y_obs', p=p, observed=y_train_bin)
    trace_gaussian = pm.sample(draws=1000, tune=500, cores=2, 
                                random_seed=RANDOM_STATE, return_inferencedata=True)

# True posterior predictive: E[sigmoid(eta)] over posterior samples,
# not sigmoid(E[eta]) which is just a plug-in / MAP approximation
post    = trace_gaussian.posterior
beta_s  = post['beta'].stack(s=('chain', 'draw')).values      # (N_COMPONENTS, S)
a_s     = post['intercept'].stack(s=('chain', 'draw')).values  # (S,)
eta_s   = X_test_pca @ beta_s + a_s                           # (N_test, S)
probs_s = 1 / (1 + np.exp(-eta_s))                            # (N_test, S)
probs      = probs_s.mean(axis=1)   # posterior predictive mean
probs_std  = probs_s.std(axis=1)    # predictive uncertainty
preds = (probs > 0.5).astype(int)

bayesian_results['Gaussian Prior'] = {
    'accuracy':  accuracy_score(y_test_bin, preds),
    'log_loss':  log_loss(y_test_bin, probs),
    'trace':     trace_gaussian,
    'probs':     probs,
    'probs_std': probs_std,
    'beta_mean': post['beta'].mean(dim=['chain', 'draw']).values,
    'beta_std':  post['beta'].std(dim=['chain', 'draw']).values,
}

print(f"Test Accuracy: {bayesian_results['Gaussian Prior']['accuracy']:.4f}")
print(f"Log-Loss:     {bayesian_results['Gaussian Prior']['log_loss']:.4f}")

In [ ]:
# (ii) Laplace Prior
print("Fitting: Bayesian Logistic Regression with Laplace Prior")

with pm.Model() as model_laplace:
    intercept = pm.Normal('intercept', mu=0, sigma=10)
    beta = pm.Laplace('beta', mu=0, b=1, shape=N_COMPONENTS)
    eta = intercept + pm.math.dot(X_train_pca, beta)
    p = pm.math.sigmoid(eta)
    y_obs = pm.Bernoulli('y_obs', p=p, observed=y_train_bin)
    trace_laplace = pm.sample(draws=1000, tune=500, cores=2,
                               random_seed=RANDOM_STATE, return_inferencedata=True)

# True posterior predictive: E[sigmoid(eta)] over posterior samples,
# not sigmoid(E[eta]) which is just a plug-in / MAP approximation
post    = trace_laplace.posterior
beta_s  = post['beta'].stack(s=('chain', 'draw')).values      # (N_COMPONENTS, S)
a_s     = post['intercept'].stack(s=('chain', 'draw')).values  # (S,)
eta_s   = X_test_pca @ beta_s + a_s                           # (N_test, S)
probs_s = 1 / (1 + np.exp(-eta_s))                            # (N_test, S)
probs      = probs_s.mean(axis=1)   # posterior predictive mean
probs_std  = probs_s.std(axis=1)    # predictive uncertainty
preds = (probs > 0.5).astype(int)

bayesian_results['Laplace Prior'] = {
    'accuracy':  accuracy_score(y_test_bin, preds),
    'log_loss':  log_loss(y_test_bin, probs),
    'trace':     trace_laplace,
    'probs':     probs,
    'probs_std': probs_std,
    'beta_mean': post['beta'].mean(dim=['chain', 'draw']).values,
    'beta_std':  post['beta'].std(dim=['chain', 'draw']).values,
}

print(f"Test Accuracy: {bayesian_results['Laplace Prior']['accuracy']:.4f}")
print(f"Log-Loss:     {bayesian_results['Laplace Prior']['log_loss']:.4f}")

In [ ]:
# (Optional) Horseshoe Prior
# Non-centered parameterization (z * lam * tau) reduces divergences with NUTS
print("Fitting: Bayesian Logistic Regression with Horseshoe Prior")

with pm.Model() as model_horseshoe:
    tau = pm.HalfCauchy('tau', beta=1)
    lam = pm.HalfCauchy('lam', beta=1, shape=N_COMPONENTS)
    intercept = pm.Normal('intercept', mu=0, sigma=10)
    # Non-centered: z ~ N(0,1) auxiliary; beta = z * lam * tau
    z    = pm.Normal('z', 0, 1, shape=N_COMPONENTS)
    beta = pm.Deterministic('beta', z * lam * tau)
    eta = intercept + pm.math.dot(X_train_pca, beta)
    p = pm.math.sigmoid(eta)
    y_obs = pm.Bernoulli('y_obs', p=p, observed=y_train_bin)
    trace_horseshoe = pm.sample(draws=1000, tune=500, cores=2,
                                 random_seed=RANDOM_STATE, return_inferencedata=True,
                                 target_accept=0.9)

# True posterior predictive: E[sigmoid(eta)] over posterior samples,
# not sigmoid(E[eta]) which is just a plug-in / MAP approximation
post    = trace_horseshoe.posterior
beta_s  = post['beta'].stack(s=('chain', 'draw')).values      # (N_COMPONENTS, S)
a_s     = post['intercept'].stack(s=('chain', 'draw')).values  # (S,)
eta_s   = X_test_pca @ beta_s + a_s                           # (N_test, S)
probs_s = 1 / (1 + np.exp(-eta_s))                            # (N_test, S)
probs      = probs_s.mean(axis=1)   # posterior predictive mean
probs_std  = probs_s.std(axis=1)    # predictive uncertainty
preds = (probs > 0.5).astype(int)

bayesian_results['Horseshoe Prior'] = {
    'accuracy':  accuracy_score(y_test_bin, preds),
    'log_loss':  log_loss(y_test_bin, probs),
    'trace':     trace_horseshoe,
    'probs':     probs,
    'probs_std': probs_std,
    'beta_mean': post['beta'].mean(dim=['chain', 'draw']).values,
    'beta_std':  post['beta'].std(dim=['chain', 'draw']).values,
}

print(f"Test Accuracy: {bayesian_results['Horseshoe Prior']['accuracy']:.4f}")
print(f"Log-Loss:     {bayesian_results['Horseshoe Prior']['log_loss']:.4f}")

In [ ]:
# Bayesian Results Summary
print("\nBayesian Results Summary:")
print("-" * 50)
for name, res in bayesian_results.items():
    print(f"{name}:")
    print(f"  Accuracy: {res['accuracy']:.4f}")
    print(f"  Mean |beta|: {np.mean(np.abs(res['beta_mean'])):.4f}")
    print(f"  Mean posterior std: {np.mean(res['beta_std']):.4f}")

In [ ]:
# Shrinkage comparison
fig, ax = plt.subplots(figsize=(10, 5))

for name, res in bayesian_results.items():
    sorted_coefs = np.sort(np.abs(res['beta_mean']))[::-1]
    ax.plot(sorted_coefs, label=name, linewidth=2)

ax.set_xlabel('Coefficient Rank')
ax.set_ylabel('|Coefficient| (Sorted)')
ax.set_title('Shrinkage Pattern Across Priors')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Uncertainty for difficult examples — ranked by posterior predictive std,
# which is a genuinely Bayesian notion of difficulty
print("\nUncertainty for Difficult Examples (ranked by posterior predictive std):")
for name, res in bayesian_results.items():
    probs      = res['probs']
    std_scores = res['probs_std']
    difficult  = np.argsort(std_scores)[-5:]  # highest posterior std = most uncertain
    print(f"\n{name}:")
    for idx in difficult:
        print(f"  Example {idx}: P(class=1)={probs[idx]:.3f}, "
              f"Pred Std={std_scores[idx]:.3f}, True={y_test_bin[idx]}")

---
## Part D: Compare and Contrast

In [ ]:
# Final Summary Table
freq_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Type': 'Frequentist',
    'Test Accuracy': [r['accuracy'] for r in results.values()],
    'Log-Loss': [r.get('log_loss', np.nan) for r in results.values()]
})

bayes_df = pd.DataFrame({
    'Model': list(bayesian_results.keys()),
    'Type': 'Bayesian',
    'Test Accuracy': [r['accuracy'] for r in bayesian_results.values()],
    'Log-Loss': [r.get('log_loss', np.nan) for r in bayesian_results.values()]
})

final_df = pd.concat([freq_df, bayes_df], ignore_index=True)
print(final_df.to_string(index=False))

### Discussion

**1. Best Classifier:**
- RBF SVM typically achieves highest accuracy due to non-linear decision boundaries
- Regularized methods (Ridge/Lasso) outperform unpenalized logistic regression

**2. Accuracy vs. Probabilistic Fit:**
- Log-loss measures calibration, not just accuracy
- SVMs don't output probabilities; logistic models do
- Bayesian methods provide full posterior uncertainty

**3. Ridge vs. Lasso:**
- Ridge shrinks uniformly; Lasso induces sparsity
- Lasso enables feature selection (interpretability)
- Ridge more stable with correlated features

**4. SVM vs. Logistic Regression:**
- SVM maximizes margin (geometric); logistic maximizes likelihood (probabilistic)
- SVM uses support vectors; logistic uses all points
- Logistic outputs calibrated probabilities

**5. Bayesian Analogues:**
- Gaussian prior = Ridge: both shrink uniformly
- Laplace prior = Lasso: both induce sparsity
- Horseshoe: adaptive shrinkage per coefficient

**6. Role of Regularization:**
- Prevents overfitting (especially when p >> n)
- Controls complexity; improves generalization
- Bayesian view: incorporates prior knowledge

In [ ]:
# Comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy comparison
all_names = list(results.keys()) + [f"Bayes: {k}" for k in bayesian_results.keys()]
all_accs = [r['accuracy'] for r in results.values()] + [r['accuracy'] for r in bayesian_results.values()]
colors = ['steelblue'] * len(results) + ['coral'] * len(bayesian_results)

axes[0].barh(all_names, all_accs, color=colors)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Model Comparison')
# Autoscale so bars are never silently clipped for low-accuracy models
axes[0].set_xlim([max(0.0, min(all_accs) - 0.02), 1.0])

# Log-loss comparison
log_losses = {k: v['log_loss'] for k, v in results.items() if v['log_loss'] is not None}
axes[1].barh(list(log_losses.keys()), list(log_losses.values()), color='steelblue')
axes[1].set_xlabel('Log-Loss')
axes[1].set_title('Probabilistic Fit')

plt.tight_layout()
plt.show()

In [ ]:
print("\nProject complete.")